# 🚀 Physics Chatbot — TPU v5e-1 Training Notebook

This notebook walks you through the **complete training pipeline** on Google Colab TPU v5e-1:

1. Mount Google Drive & setup project
2. Install dependencies (pinned for TPU v5e)
3. Verify TPU access
4. Login to HuggingFace (for gated models)
5. Download & prepare datasets
6. **Train with LoRA** (cosine LR, early stopping, auto-checkpoints)
7. Export adapter to Drive
8. Optional: merge adapter + smoke test
9. Visualize training with TensorBoard

### Prerequisites
- **Runtime → Change runtime type → TPU** (v5e-1)
- Default model: Qwen/Qwen3-4B
- If you switch to Gemma 3: accept license at https://huggingface.co/google/gemma-3-4b-it
- Have your HF token ready: https://huggingface.co/settings/tokens
- Upload `physics-chatbot-finetune` folder to Google Drive


## 1️⃣ Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2️⃣ Navigate to Project

Upload the `physics-chatbot-finetune` folder to your Google Drive root, or adjust the path below.

In [ ]:
import os

# Change this if your project is in a different Drive location
PROJECT_DIR = '/content/drive/MyDrive/physics-chatbot-finetune'

if not os.path.isdir(PROJECT_DIR):
    # Try alternate common locations
    alternatives = [
        '/content/drive/MyDrive/SaraLLM/physics-chatbot-finetune',
        '/content/drive/MyDrive/projects/physics-chatbot-finetune',
    ]
    found = False
    for alt in alternatives:
        if os.path.isdir(alt):
            PROJECT_DIR = alt
            found = True
            break
    if not found:
        raise FileNotFoundError(
            f"Project not found at {PROJECT_DIR}. "
            "Please upload the physics-chatbot-finetune folder to your Google Drive "
            "and update PROJECT_DIR above."
        )

os.chdir(PROJECT_DIR)
print(f"✅ Working directory: {os.getcwd()}")
print(f"✅ Contents: {os.listdir('.')}")

## 3️⃣ Install Dependencies

This installs PyTorch 2.6.x + XLA for TPU v5e, along with all training dependencies.

In [ ]:
!python -m pip install --upgrade pip -q
!pip install -r requirements_tpu.txt -q 2>&1 | tail -5
print("\n✅ Dependencies installed!")

## 4️⃣ Verify TPU Access

This cell sets the PJRT runtime and confirms the TPU is working.

In [ ]:
import os
os.environ['PJRT_DEVICE'] = 'TPU'

import torch
import torch_xla
import torch_xla.core.xla_model as xm

device = xm.xla_device()
print(f"✅ XLA Device: {device}")
print(f"✅ Hardware: {xm.xla_device_hw(device)}")
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ PyTorch/XLA: {torch_xla.__version__}")

# Quick memory check
try:
    mem_info = xm.get_memory_info(device)
    total_gb = mem_info.get('kb_total', 0) * 1024 / (1024**3)
    print(f"✅ TPU HBM: {total_gb:.1f} GB total")
except Exception:
    print("⚠️ Could not query TPU memory (this is normal on some runtimes)")

# Smoke test: create and operate on a TPU tensor
t = torch.randn(3, 3, device=device)
result = t @ t.T
print(f"✅ TPU compute test passed! (3x3 matmul on TPU)")

## 5️⃣ HuggingFace Login

Required for gated models like **Gemma 3** and **Llama 3.2**. Skip if using an ungated model like Qwen3.

In [ ]:
from huggingface_hub import login

# Option 1: Interactive login (paste token when prompted)
login()

# Option 2: Direct token (uncomment and paste your token)
# login(token="hf_YOUR_TOKEN_HERE")

## 6️⃣ Check Environment

Validates that all packages, paths, and runtime are correctly configured.

In [ ]:
!python scripts/00_check_env.py --config config.yaml

## 7️⃣ Download Datasets

Downloads all enabled datasets from `config.yaml` with license-aware manifesting.

In [ ]:
# First, see what sources are configured
!python scripts/01_download_datasets.py --config config.yaml --list-sources

In [ ]:
# Now download them
!python scripts/01_download_datasets.py --config config.yaml

## 8️⃣ Prepare Training Data

Cleans, deduplicates, and converts raw data into chat-format JSONL with train/val/test splits.

In [ ]:
!python scripts/02_prepare_dataset.py --config config.yaml

In [ ]:
# Quick data sanity check
import json

for split_name in ['train', 'validation', 'test']:
    path = f'data/final/{split_name}.jsonl'
    if os.path.exists(path):
        with open(path) as f:
            count = sum(1 for _ in f)
        print(f"✅ {split_name}: {count} examples")
    else:
        print(f"❌ {split_name}: file not found")

# Show the data report
report_path = 'outputs/logs/data_report.md'
if os.path.exists(report_path):
    with open(report_path) as f:
        print(f.read())

## 9️⃣ 🚀 Train on TPU v5e-1

This is the main training cell. Features:
- **Auto-resume**: if Colab disconnects and you re-run, it picks up from the last checkpoint
- **Early stopping**: stops if eval loss plateaus for 5 eval steps
- **Cosine LR schedule**: better convergence than linear decay
- **TensorBoard logging**: visualize loss curves afterwards
- **Emergency save**: if training crashes, it saves what it can

### Current Config
| Setting | Value |
|---------|-------|
| Model | Qwen/Qwen3-4B |
| LoRA rank | 32 |
| Max seq length | 1024 |
| Effective batch size | 8 |
| Learning rate | 1e-4 (cosine) |
| Epochs | 3 |
| Save every | 50 steps |

In [ ]:
# Set PJRT before training (redundant but safe)
import os
os.environ['PJRT_DEVICE'] = 'TPU'

!python scripts/03_train_tpu.py --config config.yaml

## 🔟 Save Adapter to Google Drive

Copy the trained adapter and metrics to a safe location on Drive.

In [ ]:
import shutil

EXPORT_DIR = '/content/drive/MyDrive/physics-chatbot-exports'
os.makedirs(EXPORT_DIR, exist_ok=True)

# Copy adapter
adapter_src = 'outputs/adapters/final'
adapter_dst = os.path.join(EXPORT_DIR, 'adapter_final')
if os.path.isdir(adapter_src):
    if os.path.isdir(adapter_dst):
        shutil.rmtree(adapter_dst)
    shutil.copytree(adapter_src, adapter_dst)
    print(f"✅ Adapter saved to: {adapter_dst}")
else:
    print(f"❌ Adapter not found at {adapter_src}")

# Copy training logs
for log_file in ['train_metrics.json', 'train_summary.md', 'data_report.md']:
    src = f'outputs/logs/{log_file}'
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(EXPORT_DIR, log_file))
        print(f"✅ Copied {log_file}")

print(f"\n📁 All exports saved to: {EXPORT_DIR}")

## 1️⃣1️⃣ Optional: Merge Adapter

Merges LoRA weights into the base model for standalone deployment. Only needed if you want a single model directory without requiring the base model + adapter at inference time.

In [ ]:
# Uncomment to merge:
# !python scripts/04_merge_adapter.py --config config.yaml --adapter outputs/adapters/final --output-dir outputs/merged_model
print("⏭️ Merge step skipped (uncomment above to run)")

## 1️⃣2️⃣ Quick Inference Smoke Test

Test the trained model with a physics question to verify it works.

In [ ]:
import sys
sys.path.insert(0, '.')

from src.inference import load_chat_model, generate_chat_response
from src.train_utils import load_config, get_system_prompt

config = load_config('config.yaml')
print("Loading model + adapter...")
model, tokenizer, runtime = load_chat_model(
    base_model_name=config['base_model'],
    adapter_path='outputs/adapters/final',
    trust_remote_code=True,
)
print(f"✅ Model loaded on: {runtime.get('accelerator')}")

# Test prompts
test_questions = [
    "Explain Newton's second law and define every variable.",
    "What is the relationship between wavelength, frequency, and wave speed?",
    "Explain the first law of thermodynamics.",
]

for question in test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    result = generate_chat_response(
        model=model, tokenizer=tokenizer,
        model_name=config['base_model'],
        messages=[{'role': 'user', 'content': question}],
        temperature=0.7, max_new_tokens=300,
        system_prompt=get_system_prompt(config),
    )
    print(f"A: {result['text']}")
    print(f"[{result['completion_tokens']} tokens generated]")

## 1️⃣3️⃣ Visualize Training (TensorBoard)

View training and eval loss curves.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/logs/tensorboard

## 1️⃣4️⃣ Run Evaluation (Optional)

Run the physics evaluation suite with domain probes and held-out scoring.

In [ ]:
# Uncomment to run full evaluation:
# !python scripts/07_eval_physics.py --config config.yaml --adapter outputs/adapters/final
print("⏭️ Eval step skipped (uncomment above to run)")

---

## 🎉 Done!

Your trained adapter is saved at:
- **Colab**: `outputs/adapters/final`
- **Google Drive**: `/content/drive/MyDrive/physics-chatbot-exports/adapter_final`

### Next Steps
1. Download the adapter from Drive to your local machine
2. Run `scripts/05_chat_cli.py` locally for interactive chat
3. Run `scripts/06_serve_openai_api.py` to serve via OpenAI-compatible API
4. Connect to ComfyUI using the instructions in `comfyui/README_COMFYUI.md`
